In [18]:
from scipy.io.wavfile import write
from pydub import AudioSegment
import numpy as np
from datasets import DatasetDict
from datasets import load_dataset
import os
import re
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor


In [19]:
def save_to_json_file(data, filename):

    # Convert the list of dictionaries to a JSON string
    json_string = json.dumps(data, indent=4, ensure_ascii=False)
    
    # Write the JSON string to a file
    with open(filename, 'w') as file:
        file.write(json_string)

def read_json_from_file(filename):

    with open(filename, 'r') as file:
        data = json.load(file)
    
    return data


def count_all_occurrences(lst):
    return Counter(lst)

In [20]:
ds = load_dataset("espnet/wikitongues")


Found cached dataset parquet (/home/hu/.cache/huggingface/datasets/espnet___parquet/espnet--wikitongues-0f072ef9c9629185/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)


  0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'audio'],
        num_rows: 820
    })
})

In [22]:
# dataset = ds['train'].select(range(100))
dataset = ds['train']

In [23]:
dataset

Dataset({
    features: ['id', 'audio'],
    num_rows: 820
})

In [24]:
all_file_path_list = list(map(lambda x : x["audio"]["path"], dataset))

In [25]:
# Using Regex to extract langauge names:

In [26]:

def extract_languages(file_name):
    # Search for "speaking <languages>", "signing <languages>", "in <language>", "<language> language",
    # or "<language1> and <language2>, casually spoken"
    match = re.search(
        r"(speaking|Speaking|signing|Signing) ([\w\s,‘’'-]+)|in ([\w\s'-]+?) ｜|([\w\s-]+) language|([\w\s]+(?: and [\w\s]+)*), casually spoken",
        file_name,
        re.IGNORECASE
    )
    if match:
        # Combine matches from all patterns
        languages_part = match.group(2) or match.group(3) or match.group(4) or match.group(5)
        # Split by commas and "and" or "＆" for multiple languages
        languages = re.split(r", | and | & ", languages_part)
        # Strip any leading/trailing whitespace
        return [lang.strip() for lang in languages]
    return []


In [27]:
all_language_name_list = list(map(extract_languages,  all_file_path_list))

In [28]:
# Using GPT4 to extract langauge names


In [29]:
# prompt_text = """
# You are a language extraction assistant. Your task is to extract the names of languages mentioned in a list of file names. The languages may appear after keywords like "speaking," "in," or directly before words like "language." Additionally, the names of the languages may be separated by "and" or commas.

# Here are the rules:
# 1. Extract the names of the languages mentioned after:
#    - The word "speaking" (case insensitive).
#    - The word "in" (if followed by a language).
# 2. If the language name ends with "language," only extract the language name (e.g., "Romance language" → "Romance").
# 3. If multiple languages are mentioned (e.g., "Karankawa and English"), extract them as a list of individual languages.
# 4. Ignore any unrelated text such as "Wikitongues," identifiers in square brackets, or other descriptive phrases.

# ### Examples

# #### Input and Output Examples
# 1. **Input**: `WIKITONGUES： Victorinus speaking Beahe [6htms5vFurs].wav`  
#    **Output**: `['Beahe']`

# 2. **Input**: `WIKITONGUES： Sebastiano speaking Italian [jjiXgRO8qDw].wav`  
#    **Output**: `['Italian']`

# 3. **Input**: `Josèp speaking Aranese Occitan ｜ Romance language ｜ Wikitongues [pdYpvY6Efos].wav`  
#    **Output**: `['Aranese Occitan']`

# 4. **Input**: `WIKITONGUES： Alex speaking Karankawa and English [pUfu40JELyM].wav`  
#    **Output**: `['Karankawa', 'English']`

# 5. **Input**: `John speaking Maasai ｜ East African language ｜ Wikitongues [etUt37dgA4o].wav`  
#    **Output**: `['Maasai']`

# 6. **Input**: `Aurora and Ademar speaking Mirandese ｜ Romance languages ｜ Wikitongues [5Uo-B9GaciQ].wav`  
#    **Output**: `['Mirandese']`

# ---

# Use these examples and rules to extract the languages from any input provided. Return the result as a Python list containing the extracted languages. Please output it into a JSON list format but nothing else.

# Here is your task:


# """



In [30]:
# import os
# import time
# from openai import OpenAI
# import openai
# import json
# from tqdm import tqdm
# from concurrent.futures import ThreadPoolExecutor
# import threading
# import datetime

# file_path = '/Users/hu/Documents/key/openai.key'

# if os.path.exists(file_path):
#     with open(file_path, 'r') as file:
#         openai_api_key = file.read().strip()
#         os.environ['OPENAI_API_KEY'] = openai_api_key
#         print("API Key has been set successfully.")
# else:
#     print("Error: The file 'openai.key' does not exist.")

# class Timer(object):
#     def __init__(self):
#         self.__start = time.time()

#     def start(self):
#         self.__start = time.time()

#     def get_time(self, restart=True, format=False):
#         end = time.time()
#         span = end - self.__start
#         if restart:
#             self.__start = end
#         if format:
#             return self.format(span)
#         else:
#             return span

#     def format(self, seconds):
#         return datetime.timedelta(seconds=int(seconds))

#     def print(self, name):
#         print(name, self.get_time())


# class OpenAIConfig:
#     def __init__(self, model='gpt-4o', temperature=1, max_tokens=256, logprobs=False, top_logprobs=5, attempt_num=10, buffer_path='./temp_buffer.jsonl'):
#         self.model = model
#         self.temperature = temperature
#         self.max_tokens = max_tokens
#         self.logprobs = logprobs
#         self.top_logprobs = top_logprobs if logprobs else None
#         self.attempt_num = attempt_num
#         self.buffer_path = buffer_path

#     def update(self, **kwargs):
#         for key, value in kwargs.items():
#             if hasattr(self, key):
#                 setattr(self, key, value)
#             else:
#                 raise KeyError(f'Invalid parameter name: {key}')



# class OpenAIRequestManager:
#     def __init__(self, config):
#         self.config = config

#         openai.api_key = os.getenv("OPENAI_API_KEY")
#         self.response_extractor = self.response_extractor
#         self.outbuf = open(self.config.buffer_path, 'a')
#         self.lock = threading.Lock()
#         self.client = OpenAI()

#     def response_extractor(self, response):
#         llm_output = response.choices[0].message.content.strip()
#         return {'output': llm_output}

#     def write_result(self, result):
#         self.lock.acquire()
#         self.outbuf.write(json.dumps(result, ensure_ascii=False) + '\n')
#         self.outbuf.flush()
#         self.lock.release()

#     def openai_api_call(self, input, system_prompt ="You are a helpful AI assistant."):
#         prompt = input["prompt"]
#         id = input["id"]


#         messages = [
#             {"role": "system", "content": system_prompt},
#             {"role": "user", "content": prompt},
#         ]
#         attempt = 0
#         wait_sec = 0.1
#         while True:
#             try:
#                 response = self.client.chat.completions.create(
#                     model=self.config.model,
#                     messages=messages,
#                     temperature= self.config.temperature,
#                     max_tokens=self.config.max_tokens,
#                     logprobs=self.config.logprobs,
#                     top_logprobs=self.config.top_logprobs,
#                 )

#                 result = {}
#                 llm_output = self.response_extractor(response)
#                 result["output"] = llm_output["output"]
#                 result['id'] = id
#                 result['input'] = prompt

#                 self.write_result(result)
#                 break
#             except Exception as e:
#                 print(e)
#                 attempt += 1
#                 if attempt >= self.config.attempt_num:
#                     return None
#                 time.sleep(wait_sec)

#     def multi_threading_openai_api_call(self, input_list, system_prompt ="You are a helpful AI assistant.", max_workers=64):
#         timer = Timer()
#         print(f"using model_{self.config.model}")
#         print('Processing quires')


#         with ThreadPoolExecutor(max_workers=max_workers) as executor:
#             results = list(
#                 tqdm(
#                     executor.map(lambda x : self.openai_api_call(x, system_prompt=system_prompt), input_list),
#                     total=len(input_list)
#                 )
#             )


#         print("Average time after {0} samples: {1}".format(len(input_list), timer.get_time(restart=False) / len(input_list)))
#         print('Processed queries')
#         return results

#     def load_result(self, get_list = False):
#         result_dic = {}
#         with open(self.config.buffer_path, 'r', encoding='utf-8') as file:
#             for line in file:
#                 json_object = json.loads(line)
#                 result_dic[json_object["id"]] = json_object

#         result_list = []
#         for this_id in sorted(result_dic.keys()):
#             result_list.append(result_dic[this_id])

#         if get_list:
#             return result_list
#         else:
#             return result_dic

In [31]:
# prompts = []
# for index, item in enumerate(all_file_path_list):
#     prompts.append(
#         {"prompt" : prompt_text +  item, "id" : "ID_" + str(index)}
#     )


# openai_config = OpenAIConfig()
# openai_manager = OpenAIRequestManager(config=openai_config)

# openai_manager.multi_threading_openai_api_call(input_list=prompts, max_workers=10)
# result = openai_manager.load_result()


In [32]:
# assert len(result) == len(prompts)

In [33]:
# gpt4_language_list = []
# for i in range(len(prompts)):
#     this_id = "ID_" + str(i)
#     gpt4_language_list.append(result[this_id]["output"])

In [34]:

# assert len(gpt4_language_list) == len(all_language_name_list)

In [36]:
# con = 0
# for gpt_result, regex_result, file_path in list(zip(gpt4_language_list, all_language_name_list, all_file_path_list))[:]:
#     regex_lan_list = regex_result
#     gpt_lan_list = json.loads(gpt_result.split("\n")[1])   
#     if regex_lan_list != gpt_lan_list:
#         con += 1
    
#         print(file_path)
#         print(regex_lan_list)
#         print(gpt_lan_list)
#         print()
    
# print(con)

In [ ]:
# GPT4 results make sense. We will use it.

In [37]:
# langauge_list = save_to_json_file(gpt4_language_list, "./language_list.json")

In [56]:
langauge_list = read_json_from_file("./language_list.json")

In [57]:

langauge_list = list(map(lambda x : json.loads(x.split("\n")[1])  , langauge_list))

In [58]:
def get_wav_duration_in_minutes(audio_dict):
    audio_array = np.array(audio_dict['audio']['array'])
    sampling_rate = audio_dict['audio']['sampling_rate']
    duration_seconds = len(audio_array) / sampling_rate
    duration_minutes = duration_seconds / 60
    return duration_minutes

# def calculate_duration_for_item(item):
#     item['duration_minutes'] = get_wav_duration_in_minutes(item)
#     return item

In [59]:
audio_minute_list = []
for item in tqdm(dataset):
    audio_minute_list.append(get_wav_duration_in_minutes(item))

100%|██████████████████████████████████████| 820/820 [00:11<00:00, 71.67it/s]


In [60]:
assert len(audio_minute_list) == len(all_file_path_list)

In [61]:
item

{'id': 'jf_00819',
 'audio': {'path': 'WIKITONGUES： John speaking English, Spanish, and Greek [22HBqPN8Oco].wav',
  'array': array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         3.05175781e-05, 3.05175781e-05, 3.05175781e-05]),
  'sampling_rate': 16000}}

In [62]:
result_list = []
for index, item in enumerate(dataset):
    result_entry = {}
    result_entry["id"] = item["id"]
    result_entry["audio_path"] = all_file_path_list[index]
    result_entry["language_list"] = langauge_list[index]
    result_entry["length_in_minute"] = audio_minute_list[index]
    result_list.append(result_entry)

In [63]:
save_to_json_file(result_list, "./result.json")